# 文本分类实例

## Step1 导入相关包

In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset

## Step2 加载数据集

In [2]:
dataset = load_dataset("csv", data_files="./ChnSentiCorp_htl_all.csv", split="train")
dataset = dataset.filter(lambda x: x["review"] is not None)
dataset

Dataset({
    features: ['label', 'review'],
    num_rows: 7765
})

## Step3 划分数据集

In [3]:
datasets = dataset.train_test_split(test_size=0.1)
datasets

DatasetDict({
    train: Dataset({
        features: ['label', 'review'],
        num_rows: 6988
    })
    test: Dataset({
        features: ['label', 'review'],
        num_rows: 777
    })
})

## Step4 创建 Dataloader

In [4]:
import torch

tokenizer = AutoTokenizer.from_pretrained("hfl/rbt3")


def process_function(examples):
    tokenized_examples = tokenizer(examples["review"], max_length=128, truncation=True)
    tokenized_examples["labels"] = examples["label"]
    return tokenized_examples


tokenized_datasets = datasets.map(
    process_function, batched=True, remove_columns=datasets["train"].column_names
)

tokenized_datasets

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/6988 [00:00<?, ? examples/s]

Map:   0%|          | 0/777 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 6988
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 777
    })
})

In [5]:
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding

trainset, validset = tokenized_datasets["train"], tokenized_datasets["test"]
trainloader = DataLoader(
    trainset, batch_size=32, shuffle=True, collate_fn=DataCollatorWithPadding(tokenizer)
)
validloader = DataLoader(
    validset,
    batch_size=64,
    shuffle=False,
    collate_fn=DataCollatorWithPadding(tokenizer),
)

In [6]:
next(enumerate(validloader))[1]

{'input_ids': tensor([[ 101, 4696,  679,  ...,  738, 7478,  102],
        [ 101, 1920, 2157,  ...,    0,    0,    0],
        [ 101, 3241,  677,  ...,    0,    0,    0],
        ...,
        [ 101, 1765, 4415,  ...,    0,    0,    0],
        [ 101, 2769, 6821,  ...,    0,    0,    0],
        [ 101, 4384, 1862,  ...,  117, 4692,  102]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1]]), 'labels': tensor([0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1,
        1, 1, 0

## Step5 创建模型及优化器

In [7]:
from torch.optim import Adam

model = AutoModelForSequenceClassification.from_pretrained("hfl/rbt3")

if torch.cuda.is_available():
    model = model.cuda()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at hfl/rbt3 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
optimizer = Adam(model.parameters(), lr=2e-5)

## Step6 训练与验证

In [9]:
def evaluate():
    model.eval()
    acc_num = 0
    with torch.inference_mode():
        for batch in validloader:
            if torch.cuda.is_available():
                batch = {k: v.cuda() for k, v in batch.items()}
            output = model(**batch)
            pred = torch.argmax(output.logits, dim=-1)
            print(f"evaluate pred: {pred.long()}, labels: {batch["labels"].long()}")
            acc_num += (pred.long() == batch["labels"].long()).float().sum()
    return acc_num / len(validset)


def train(epoch=3, log_step=10):
    global_step = 0

    for ep in range(epoch):
        model.train()
        # 每个 epoch 训练全部数据
        for batch in trainloader:
            if torch.cuda.is_available():
                batch = {k: v.cuda() for k, v in batch.items()}

            optimizer.zero_grad()
            output = model(**batch)
            output.loss.backward()
            optimizer.step()

            if global_step % log_step == 0:
                print(
                    f"ep: {ep}, global_step: {global_step}, loss: {output.loss.item()}"
                )

            global_step += 1

        acc = evaluate()
        print(f"ep: {ep}, acc: {acc}")

## Step7 模型训练

In [10]:
train()

ep: 0, global_step: 0, loss: 0.8316254615783691
ep: 0, global_step: 10, loss: 0.6797478795051575
ep: 0, global_step: 20, loss: 0.4850689768791199
ep: 0, global_step: 30, loss: 0.5484188795089722
ep: 0, global_step: 40, loss: 0.5893192291259766
ep: 0, global_step: 50, loss: 0.5090286731719971
ep: 0, global_step: 60, loss: 0.41484788060188293
ep: 0, global_step: 70, loss: 0.34550201892852783
ep: 0, global_step: 80, loss: 0.31966832280158997
ep: 0, global_step: 90, loss: 0.30499356985092163
ep: 0, global_step: 100, loss: 0.22987614572048187
ep: 0, global_step: 110, loss: 0.2669928967952728
ep: 0, global_step: 120, loss: 0.2294473797082901
ep: 0, global_step: 130, loss: 0.2709316611289978
ep: 0, global_step: 140, loss: 0.46110206842422485
ep: 0, global_step: 150, loss: 0.3245270550251007
ep: 0, global_step: 160, loss: 0.21109719574451447
ep: 0, global_step: 170, loss: 0.1652633100748062
ep: 0, global_step: 180, loss: 0.14627109467983246
ep: 0, global_step: 190, loss: 0.35995006561279297
ep

## Step8 模型预测

In [11]:
sen = "我觉得这家酒店不错，饭很好吃！"
id2_label = {0: "差评！", 1: "好评！"}
model.eval()
with torch.inference_mode():
    inputs = tokenizer(sen, return_tensors="pt")
    inputs = {k: v.cuda() for k, v in inputs.items()}
    output = model(**inputs)
    print(f"output: {output}")
    logits = output.logits
    print(f"logits: {logits}")
    pred = torch.argmax(logits, dim=-1)
    print(f"pred: {pred}")
    print(f"输入: {sen}\n模型预测结果:{id2_label.get(pred.item())}")

output: SequenceClassifierOutput(loss=None, logits=tensor([[-2.1220,  1.9170]], device='cuda:0'), hidden_states=None, attentions=None)
logits: tensor([[-2.1220,  1.9170]], device='cuda:0')
pred: tensor([1], device='cuda:0')
输入: 我觉得这家酒店不错，饭很好吃！
模型预测结果:好评！


In [12]:
from transformers import pipeline

model.config.id2label = id2_label
pipe = pipeline("text-classification", model=model, tokenizer=tokenizer, device=0)

Device set to use cuda:0


In [13]:
pipe(sen)

[{'label': '好评！', 'score': 0.982689380645752}]